# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ujjwalupreti/flyrank-internship-capstone/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of Analysis:** One row represents exactly one unique webpage (identified by `content_id`) for a specific client (`client_id`).

**Time Window:** The data evaluates these pages over a single, fixed one-month window (e.g., March 2026) to prevent timeline overlap between our historical features and future labels.



In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

* **Features:** `content_age_days`, `impressions`, `clicks`, `ctr`, `content_type`
* **Label (Target):** `is_declining_label` (a binary yes/no calculated from the `trend_direction` being "down").

* **Context:** `client_id`, `content_id` (used for grouping and tracking, never for modeling).
* **Excluded:** `trend_pct` and `trend_direction`. Because the target label is computed directly from these columns, including them in the feature set would create label-derived leakage.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [8]:
# Install DuckDB and load the Hugging Face token securely
%pip -q install duckdb huggingface_hub
import duckdb
from google.colab import userdata
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score

# 1. Connect and Authenticate
con = duckdb.connect()
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

# Target the daily performance fact table
REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

print("--- Data Dictionary (Schema Check) ---")
schema = con.sql(f"DESCRIBE SELECT * FROM {REL}").df()
print(schema['column_name'].tolist())
print("\n--------------------------------------\n")

print("--- Query 1: Row Count & Date Span ---")
q1 = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date
    FROM {REL}
""").df()
print(q1)

print("\n--- Query 2: Grain Verification (1 row = 1 page per day) ---")
# Updated to content_hash_id and grouped by report_date
q2 = con.sql(f"""
    SELECT COUNT(*) as duplicate_rows
    FROM (
        SELECT content_hash_id, report_date, COUNT(*)
        FROM {REL}
        GROUP BY content_hash_id, report_date
        HAVING COUNT(*) > 1
    )
""").df()
print(q2)

print("\n--- Query 3: Availability (Rows with active impressions) ---")
# Swapped to gsc_impressions based on the schema
q3 = con.sql(f"""
    SELECT COUNT(*) AS active_rows
    FROM {REL}
    WHERE gsc_impressions > 0
""").df()
print(q3)

print("\n--- Feature Frame & The Trap ---")
# Aggregating the daily rows up to the page level to build our feature frame
df_features = con.sql(f"""
    SELECT
        content_hash_id,
        MAX(gsc_impressions) AS max_impressions_last_30d,
        SUM(gsc_clicks) AS clicks_last_30d,
        -- Creating a mock label based on traffic drops for the trap
        CASE WHEN SUM(gsc_clicks) < 10 THEN 1 ELSE 0 END AS is_declining_label,
        -- The leaky feature (a mathematical derivative of the label)
        CASE WHEN SUM(gsc_clicks) < 10 THEN 'down' ELSE 'stable' END AS trend_direction
    FROM {REL}
    WHERE gsc_impressions > 0
    GROUP BY content_hash_id
    LIMIT 5000
""").df()

print(f"Feature frame built with {len(df_features)} rows.")

# THE TRAP: Label-derived feature
df_features['leaky_feature'] = np.where(df_features['trend_direction'] == 'down', 1, 0)
# We use clicks_last_30d and the leaky feature
features_with_leak = ['clicks_last_30d', 'leaky_feature']

clf_leaky = DecisionTreeClassifier(random_state=42)
clf_leaky.fit(df_features[features_with_leak], df_features['is_declining_label'])
preds_leaky = clf_leaky.predict(df_features[features_with_leak])
score = precision_score(df_features['is_declining_label'], preds_leaky)

print(f"\nTrap Score (with leaky feature): Precision = {score:.2f}")
print("Lesson: The score is perfectly 1.0 because the feature 'leaky_feature' is just the label in disguise. The model learned the rule, not the world.")

# Removing the leaked column to keep the honest frame
df_features = df_features.drop(columns=['trend_direction', 'leaky_feature'])

--- Data Dictionary (Schema Check) ---
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']

--------------------------------------

--- Query 1: Row Count & Date Span ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows start_date   end_date
0     9841378 2026-03-01 2026-03-31

--- Query 2: Grain Verification (1 row = 1 page per day) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   duplicate_rows
0               0

--- Query 3: Availability (Rows with active impressions) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   active_rows
0      3611061

--- Feature Frame & The Trap ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame built with 5000 rows.

Trap Score (with leaky feature): Precision = 1.00
Lesson: The score is perfectly 1.0 because the feature 'leaky_feature' is just the label in disguise. The model learned the rule, not the world.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*


This dataset strictly relies on historical Google Search Console (GSC) performance and page age. It **cannot** tell us about external market factors (e.g., a competitor launching a better page, or a Google core algorithm update shifting search intent). Because it relies on historical interactions, pages with zero initial impressions are functionally invisible to this specific analysis slice.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.